Mandatory imports for our Data Cleaning
Read .csv dataset into pandas DataFrame

In [16]:
import pandas as pd
import numpy as np
import matplotlib as plt

In [17]:
# Adjust display options to see all columns
pd.set_option('display.max_columns', None)

# Read dataset from .csv file
df = pd.read_csv('../data/raw/bkk_positions.csv', dtype=str)

print(f'Initial dataset shape: {df.shape}')
df.head()

Initial dataset shape: (592993, 18)


,ingested_at,id,vehicle.vehicle.id,vehicle.vehicle.label,vehicle.vehicle.licensePlate,vehicle.position.bearing,vehicle.position.latitude,vehicle.position.longitude,vehicle.timestamp,vehicle.currentStatus,vehicle.position.speed,vehicle.trip.tripId,vehicle.trip.routeId,vehicle.trip.startDate,vehicle.trip.scheduleRelationship,vehicle.stopId,vehicle.currentStopSequence,vehicle.congestionLevel
0,2026-04-16 19:05:09.432211,VehiclePosition-BKK_1,1,Deák F. tér M (City centre),TSM002,218.0,47.432663,19.261341,1776333261,IN_TRANSIT_TO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-04-16 19:05:09.438127,VehiclePosition-BKK_100,100,NaN,AADI603,72.0,47.47466,19.02789,1776360439,IN_TRANSIT_TO,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-04-16 19:05:09.441053,VehiclePosition-BKK_1000,1000,"Csepel, Soroksári rév",AOJM876,316.0,47.456635,19.116358,1776366299,IN_TRANSIT_TO,0.0,D12367625,1480,20260416,SCHEDULED,F01370,11.0,NaN
3,2026-04-16 19:05:09.444185,VehiclePosition-BKK_1003,1003,Kőbánya-Kispest M,AOJM877,289.0,47.422478,19.069391,1776366302,IN_TRANSIT_TO,3.601108,D12367595,1480,20260416,SCHEDULED,F04185,13.0,NaN
4,2026-04-16 19:05:09.447917,VehiclePosition-BKK_1005,1005,NaN,AOJM879,266.0,47.443203,19.082645,1776324825,IN_TRANSIT_TO,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df['latitude'] = pd.to_numeric(df['vehicle.position.latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['vehicle.position.longitude'], errors='coerce')

df = df.dropna(subset=['latitude', 'longitude'])

df = df[(df['latitude'] != 0.0) & (df['longitude'] != 0.0)]

valid_statuses = ['IN_TRANSIT_TO', 'STOPPED_AT', 'INCOMING_AT']

df = df[df['vehicle.currentStatus'].isin(valid_statuses)]

print("Cleaned Vehicle Status Distribution:")
print(df['vehicle.currentStatus'].value_counts())
print(f"Final dataset shape before saving: {df.shape}")


Cleaned Vehicle Status Distribution:
vehicle.currentStatus
IN_TRANSIT_TO    244028
STOPPED_AT        76444
Name: count, dtype: int64
Final dataset shape before saving: (320472, 20)


In [19]:
# Drop unnecessary raw columns to save space
cols_to_drop = ['vehicle.position.latitude', 'vehicle.position.longitude', 'delay'] # Add others as needed
df_clean = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Save the cleaned dataset
processed_path = '../data/processed/bkk_cleaned.csv'
df_clean.to_csv(processed_path, index=False)

print(f"Cleaned dataset saved to {processed_path} with shape {df_clean.shape}")

Cleaned dataset saved to ../data/processed/bkk_cleaned.csv with shape (320472, 18)


In [20]:
import ast
import re
import pandas as pd
import numpy as np

# Load your physical positions dataset first
print("Loading Positions Data...")
df = pd.read_csv('../data/raw/bkk_positions.csv', dtype=str) # Update with your actual path/variable
df['vehicle.timestamp'] = pd.to_numeric(df['vehicle.timestamp'], errors='coerce')

# ---------------------------------------------------------
# PROCESS DELAYS IN CHUNKS TO PREVENT OOM FREEZES
# ---------------------------------------------------------
print("Processing Trip Updates (ETA Data) in chunks...")

def process_delays_chunk(chunk):
    if 'tripUpdate.trip.tripId' in chunk.columns:
        chunk = chunk.rename(columns={'tripUpdate.trip.tripId': 'vehicle.trip.tripId'})

    def safe_parse_list(val):
        if pd.isna(val):
            return []
        try:
            clean_val = re.sub(r'\bnan\b', 'None', str(val))
            return ast.literal_eval(clean_val)
        except Exception:
            return []

    chunk['tripUpdate.stopTimeUpdate'] = chunk['tripUpdate.stopTimeUpdate'].apply(safe_parse_list)
    
    # Explode the stops
    exploded = chunk.explode('tripUpdate.stopTimeUpdate').dropna(subset=['tripUpdate.stopTimeUpdate'])

    # Extract the absolute arrival TIME
    def extract_arrival_time(stop_dict):
        if isinstance(stop_dict, dict):
            arrival = stop_dict.get('arrival', {})
            if isinstance(arrival, dict):
                return arrival.get('time', np.nan)
        return np.nan

    exploded['arrival_timestamp'] = exploded['tripUpdate.stopTimeUpdate'].apply(extract_arrival_time)

    # Clean and isolate the closest upcoming ETA
    exploded = exploded.dropna(subset=['arrival_timestamp'])
    exploded['arrival_timestamp'] = exploded['arrival_timestamp'].astype(float)
    exploded = exploded.sort_values('arrival_timestamp')
    
    # Keep only the closest upcoming ETA for each bus IN THIS CHUNK
    return exploded[['vehicle.trip.tripId', 'arrival_timestamp']]

chunk_size = 25000  # Adjust if it still uses too much RAM
eta_chunks = []

# Iterate through the large CSV in memory-safe batches
chunks_amount = -1 # -1 for max amount
for i, chunk in enumerate(pd.read_csv('../data/raw/bkk_delays.csv', dtype=str, chunksize=chunk_size)):
    print(f"\rProcessing chunk {i+1}...      ", end="", flush=True)
    processed_chunk = process_delays_chunk(chunk)
    eta_chunks.append(processed_chunk)

    if chunks_amount != -1 and i + 1 >= chunks_amount:
        print(f"Test limit reached. Stopping after {chunks_amount} chunks.")
        break
print()

print("Concatenating processed chunks...")
# Combine all the tiny dataframes
eta_features = pd.concat(eta_chunks)

# DO NOT drop duplicates here! We need every stop for merge_asof.
# Just drop any lingering nulls before sorting.
print("Cleaning nulls and sorting data for merge_asof...")
df = df.dropna(subset=['vehicle.timestamp', 'vehicle.trip.tripId'])
eta_features = eta_features.dropna(subset=['arrival_timestamp', 'vehicle.trip.tripId'])

# Sort both datasets chronologically (Mandatory for merge_asof)
df = df.sort_values('vehicle.timestamp')
eta_features = eta_features.sort_values('arrival_timestamp')

# ---------------------------------------------------------
# THE GRAND MERGE (Using merge_asof)
# ---------------------------------------------------------
print("Executing asynchronous merge...")

df = pd.merge_asof(
    left=df, 
    right=eta_features, 
    by='vehicle.trip.tripId',           # Match the exact bus trip
    left_on='vehicle.timestamp',        # The time of the GPS ping
    right_on='arrival_timestamp',       # The time of the stop ETA
    direction='forward'                 # Look FORWARD in time to find the NEXT stop
)

# Drop any GPS pings that couldn't find a future stop
df = df.dropna(subset=['arrival_timestamp'])

# ---------------------------------------------------------
# CREATE THE TARGET VARIABLE
# ---------------------------------------------------------
df['seconds_to_next_stop'] = df['arrival_timestamp'] - df['vehicle.timestamp']

# Filter out extreme outliers (> 1 hour between stops is likely an API error)
df = df[(df['seconds_to_next_stop'] > 0) & (df['seconds_to_next_stop'] < 3600)]

# You can keep nrows=50000 on the CSV reader to test this quickly again!
print(f"FINAL DATASET READY FOR ML: {df.shape}")

print("Saving cleaned data to CSV...")
df.to_csv('../data/processed/bkk_cleaned.csv', index=False)
print("Done! You are ready for EDA.")


Loading Positions Data...
Processing Trip Updates (ETA Data) in chunks...
Processing chunk 45...      
Concatenating processed chunks...
Cleaning nulls and sorting data for merge_asof...
Executing asynchronous merge...
FINAL DATASET READY FOR ML: (13702, 20)
Saving cleaned data to CSV...
Done! You are ready for EDA.
